# Chat-template continuation check

Lightweight, tokenizer-only verification of the 2026-06-11 finding
(`docs/findings.md`): does this environment's transformers preserve the
trailing `\n\n` step separator through
`apply_chat_template(continue_final_message=True)`? That is the condition
for MCTS searches to produce complete trajectories ending with
"The final answer is".

Run all cells in the env you want to classify — seconds, no GPU, no vLLM.

Expected: **py311 (canonical) → all checks pass**; old stack
(transformers 4.45) → `unstripped prompt keeps separator` fails.

In [1]:
import sys
from importlib.metadata import version

print("python      ", sys.version.split()[0])
for pkg in ["transformers", "torch", "vllm"]:
    try:
        print(f"{pkg:<12}", version(pkg))
    except Exception:
        print(f"{pkg:<12}", "not installed")

python       3.11.15
transformers 4.45.2
torch        2.5.1
vllm         0.6.4


## Check 1 — template rendering with trailing separator

Assistant content ending `\n\n`, `continue_final_message=True`, under the
stock Llama template vs SAL's custom template (the one the pipeline
always uses — launchers do not override `config.custom_chat_template`).
SAL's template differs from stock in exactly one way: it does **not**
`| trim` assistant message content.

In [2]:
from transformers import AutoTokenizer
from sal.config import Config

base_dir = "/groups/chichengz/tnn/datasets"
tokenizer = AutoTokenizer.from_pretrained(f"{base_dir}/Llama3.2-1B-Instruct")
stock_template = tokenizer.chat_template
config = Config()

conv = [
    {"role": "system", "content": "solve it"},
    {"role": "user", "content": "1+1?"},
    {"role": "assistant", "content": "Step one.\n\n"},
]
results = {}
for name, template in [("stock", stock_template),
                       ("sal", config.custom_chat_template)]:
    tokenizer.chat_template = template
    try:
        rendered = tokenizer.apply_chat_template(
            conv, continue_final_message=True, tokenize=False
        )
        results[name] = rendered.endswith("\n\n")
        print(f"{name:<6} renders OK, separator preserved: {results[name]}")
    except ValueError as e:
        results[name] = None
        print(f"{name:<6} FAILS ValueError: {e}")

/home/u20/tnguyen9210/micromamba/envs/vllm1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


stock  FAILS ValueError: substring not found
sal    renders OK, separator preserved: False


## Check 2 — the actual search prompt path

Reproduces `_generate_candidates` prompt construction
(`mcts_cnt_search_v05_00_00`) for a depth-2 node whose text ends with the
step separator, in both variants: unstripped vs
`removesuffix("\n\n")`. The model only continues with a next step
(instead of emitting EOS) when the final prompt ends with `\n\n`.

In [3]:
from sal.search.utils import build_conv

tokenizer.chat_template = config.custom_chat_template
current_text = "Step one.\n\nStep two.\n\n"

prompts = {}
for variant, text in [("no_strip", current_text),
                      ("strip", current_text.removesuffix("\n\n"))]:
    convs = [build_conv("1+1?", text, config.system_prompt)]
    templated = tokenizer.apply_chat_template(
        convs,
        add_generation_prompt=False,
        continue_final_message=True,
        date_string="Aug 1 2025",
        tokenize=False,
    )[0]
    ends = templated.endswith("\n\n")
    prompts[variant] = ends
    print(f"{variant:<9} prompt ends with separator: {ends}")

2026-06-11 18:58:52,352	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


no_strip  prompt ends with separator: False
strip     prompt ends with separator: False


## Check 3 — PRM scoring tokenization

The `reward_models.py` pattern: `apply_chat_template` must return a
dict-like with `["input_ids"]`. Requires `return_dict=True` in
transformers 4.45 (fixed 2026-06-11); verifies the fixed call works in
this env.

In [4]:
prm_tok = AutoTokenizer.from_pretrained(
    f"{base_dir}/Llama3.1-8B-PRM-Deepseek-Data"
)
prm_tok.padding_side = "right"
prm_tok.pad_token = prm_tok.eos_token

prm_convs = [[{"content": "1+1? Step one.", "role": "user"},
              {"content": "+", "role": "assistant"}]]
out_dict = prm_tok.apply_chat_template(
    prm_convs, padding=True, return_dict=True, return_tensors="pt"
)
prm_ok = hasattr(out_dict, "keys") and out_dict["input_ids"].ndim == 2
print("prm return_dict pattern works:", prm_ok)

prm return_dict pattern works: True


## Summary

Passes iff this env should produce complete trajectories.

In [5]:
checks = {
    "sal template renders": results["sal"] is not None,
    "unstripped prompt keeps separator": bool(prompts["no_strip"]),
    "prm tokenization (return_dict=True) works": prm_ok,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL':<5} {name}")

assert all(checks.values()), (
    "env will NOT produce complete trajectories -- "
    "see docs/findings.md (2026-06-11)"
)
print("\nALL CHECKS PASSED -- env should produce complete trajectories")

PASS  sal template renders
FAIL  unstripped prompt keeps separator
PASS  prm tokenization (return_dict=True) works


AssertionError: env will NOT produce complete trajectories -- see docs/findings.md (2026-06-11)